# GEM_02: Model Building

In [2]:
import cobra
import pickle
import os
import numpy as np
from cobra.flux_analysis import pfba
from optlang.symbolics import Zero

## 1: Data Loading & Map Metabolites to Human-GEM Model

### 1.1: Load Human2 GEM (Human-GEM) Model

In [3]:
# Path to Human2 GEM SBML file
GEM_PATH = "/Users/maisievarcoe/Desktop/Research_Project/Scripts/Integration/GEM/pyglpk/Human-GEM/model/Human-GEM.xml"

# Load the model
model = cobra.io.read_sbml_model(GEM_PATH)

print("Model loaded:", model.name)
print("Reactions:", len(model.reactions))
print("Metabolites:", len(model.metabolites))
print("Genes:", len(model.genes))

Model loaded: Generic genome-scale metabolic model of Homo sapiens
Reactions: 12931
Metabolites: 8461
Genes: 2848


### 1.2: Load GEM Input Data from the GEM_01 Pipeline
### This is all the input data in one file to make running this pipeline easier

In [4]:
DATA_PATH = "/Users/maisievarcoe/Desktop/Research_Project/Scripts/Integration/GEM/pyglpk/Data/gem_input_data.pkl"

with open(DATA_PATH, "rb") as f:
    gem_data = pickle.load(f)

metadata        = gem_data["metadata"]
nmr             = gem_data["nmr"]
microbiome      = gem_data["microbiome"]
pathways        = gem_data["pathways"]
mean_expr       = gem_data["mean_expr"]
mean_expr_high  = gem_data["mean_expr_high"]
mean_expr_low   = gem_data["mean_expr_low"]
metabolite_map  = gem_data["metabolite_map"]
gem_metabolites = gem_data["gem_metabolites"]
common_ids      = gem_data["common_ids"]

print("Loaded GEM input data.")
print("Samples:", len(common_ids))
print("Metabolites for constraints:", gem_metabolites)


Loaded GEM input data.
Samples: 18
Metabolites for constraints: ['Succinic acid', 'Lactic acid', 'Proline', 'Lysine', 'Alanine', 'Tyrosine', 'Isoleucine', 'Tryptophan', 'Glucose', 'Arginine']


In [5]:
print("Metadata shape:", metadata.shape)
print("NMR shape:", nmr.shape)
print("Microbiome shape:", microbiome.shape)
print("Pathways shape:", pathways.shape)
print("Mean expr shape:", mean_expr.shape)
print("Metabolite map:", metabolite_map)


Metadata shape: (18, 16)
NMR shape: (18, 38)
Microbiome shape: (18, 44)
Pathways shape: (18, 250)
Mean expr shape: (23017, 11)
Metabolite map: {'Succinic acid': 'MAM02943c', 'Lactic acid': 'MAM02403c', 'Proline': 'MAM02770c', 'Lysine': 'MAM02426c', 'Alanine': 'MAM01307c', 'Tyrosine': 'MAM03101c', 'Isoleucine': 'MAM02184c', 'Tryptophan': 'MAM03089c', 'Glucose': 'MAM01965c', 'Arginine': 'MAM01365c'}


### 1.3: Create Human-GEM Exchange Reaction ID Map - to use for Constraining Model

In [11]:
# Human_GEM Exchange Reaction IDs - for applying uptake/secretion constraints
# Exchange mapping dictionary for each metabolite and its corresponding Human_GEM ID
# Use these IDs to add constraints
exchange_map = {
    "Succinic acid": "MAR09415",
    "Lactic acid":   "MAR09135",
    "Proline":       "MAR09068",
    "Lysine":        "MAR09041",
    "Alanine":       "MAR09061",
    "Tyrosine":      "MAR09064",
    "Isoleucine":    "MAR09039",
    "Tryptophan":    "MAR09045",
    "Glucose":       "MAR09034",
    "Arginine":      "MAR09066",
}

## 2: Cell-Type Specific Model Comparing High vs Low SA, for Ker1 Cells (GIMME)

### 2.1: Map Gene Symbols to Ensembl IDs

In [18]:
# Get mapping table from mygene
mean_expr.index[:10]
import sys
!{sys.executable} -m pip install mygene

import mygene

mg = mygene.MyGeneInfo()

# Query all gene symbols at once
symbols = list(mean_expr.index)
mapping = mg.querymany(symbols, scopes='symbol', fields='ensembl.gene', species='human')


  Using cached mygene-3.2.2-py2.py3-none-any.whl.metadata (10 kB)
  Using cached biothings_client-0.5.0-py3-none-any.whl.metadata (11 kB)
Using cached mygene-3.2.2-py2.py3-none-any.whl (5.4 kB)
Using cached biothings_client-0.5.0-py3-none-any.whl (51 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [mygene]


518 input query terms found dup hits:	[('LINC00115', 2), ('SLC35E2A', 2), ('TNFRSF14-AS1', 3), ('CAMTA1-DT', 2), ('RERE-AS1', 2), ('EXOSC1
5303 input query terms found no hit:	['AL627309.1', 'AL627309.5', 'AL645608.7', 'AL645608.1', 'AL390719.2', 'AL391244.2', 'AL391244.1', '


In [25]:
# Mapping disctionary of gene symbols to ensembl IDs
# Build symbol → ensembl mapping dictionary
symbol_to_ens = {}

for entry in mapping:
    if 'notfound' in entry and entry['notfound']:
        continue
    if 'ensembl' not in entry:
        continue
    
    ens = entry['ensembl']
    
    # If multiple Ensembl IDs, take the first
    if isinstance(ens, list):
        ens = ens[0]
    
    if isinstance(ens, dict) and 'gene' in ens:
        symbol_to_ens[entry['query']] = ens['gene']
len(symbol_to_ens)




17618

In [26]:
# Apply mapping to expression matrices of single cell data (and SA high/low sc data)
def convert_expr_to_ensembl(expr_df, mapping_dict):
    # Keep only genes that mapped
    expr_df = expr_df.loc[expr_df.index.intersection(mapping_dict.keys())].copy()
    
    # Rename index to Ensembl IDs
    expr_df.index = expr_df.index.map(mapping_dict)
    
    # Remove duplicates (multiple symbols → same Ensembl)
    expr_df = expr_df.groupby(expr_df.index).mean()
    
    return expr_df
    
mean_expr_ens      = convert_expr_to_ensembl(mean_expr, mapping_dict=symbol_to_ens)
mean_expr_high_ens = convert_expr_to_ensembl(mean_expr_high, mapping_dict=symbol_to_ens)
mean_expr_low_ens  = convert_expr_to_ensembl(mean_expr_low, mapping_dict=symbol_to_ens)

model_genes = {g.id for g in model.genes}

len(model_genes & set(mean_expr_ens.index))

2448

In [29]:
# Check the genes overlap in model and single cell expression data
# Use the Ensembl-mapped expression matrices
expr_high_KER1 = mean_expr_high_ens[cell_type]   # Series: Ensembl IDs → expression
expr_low_KER1  = mean_expr_low_ens[cell_type]

print("KER1 High expr genes:", expr_high_KER1.shape)
print("KER1 Low expr genes:", expr_low_KER1.shape)

model_genes = {g.id for g in model.genes}
expr_genes  = set(mean_expr_ens.index)

print("Genes in both model and expression:", len(model_genes & expr_genes))



KER1 High expr genes: (17618,)
KER1 Low expr genes: (17618,)
Genes in both model and expression: 2448


### 2.2: Define KER1 expression and NMR Group means for high vs low SA

In [30]:
cell_type = "KER1"

# KER1 expression (Ensembl IDs)
expr_high_KER1 = mean_expr_high_ens[cell_type]   # Series: Ensembl → expr
expr_low_KER1  = mean_expr_low_ens[cell_type]

print("KER1 High expr genes:", expr_high_KER1.shape)
print("KER1 Low expr genes:", expr_low_KER1.shape)

# Group means for NMR (High vs Low SA)
high_ids = metadata[metadata["SA_status"] == "High"].index
low_ids  = metadata[metadata["SA_status"] == "Low"].index

nmr_high_mean = nmr.loc[high_ids].mean()
nmr_low_mean  = nmr.loc[low_ids].mean()

print("NMR High mean shape:", nmr_high_mean.shape)
print("NMR Low mean shape:", nmr_low_mean.shape)


KER1 High expr genes: (17618,)
KER1 Low expr genes: (17618,)
NMR High mean shape: (38,)
NMR Low mean shape: (38,)


### 2.3: Build Reaction Weights from Expression (GIMME)

In [31]:
def build_reaction_weights(model, expr_series, low_thresh=0.1):
    """
    expr_series: pandas Series, index = Ensembl IDs, values = expression.
    Returns: dict {reaction_id: weight}, where:
      0.0 = high expression (no penalty)
      1.0 = low/no expression (penalise)
    """
    weights = {}
    expr_index = set(expr_series.index)

    for rxn in model.reactions:
        if not rxn.genes:
            weights[rxn.id] = 0.0
            continue

        gene_exprs = []
        for g in rxn.genes:
            if g.id in expr_index:
                gene_exprs.append(expr_series[g.id])

        if not gene_exprs:
            weights[rxn.id] = 1.0
        else:
            avg_expr = float(np.mean(gene_exprs))
            weights[rxn.id] = 0.0 if avg_expr >= low_thresh else 1.0

    return weights

weights_high_KER1 = build_reaction_weights(model, expr_high_KER1)
weights_low_KER1  = build_reaction_weights(model, expr_low_KER1)

len(weights_high_KER1), len(weights_low_KER1)


(12931, 12931)

### 2.4: Apply NMR-Based Metabolite Constraints

In [32]:
def apply_metabolite_constraints(base_model, nmr_profile, exchange_map, scale=0.01):
    """
    base_model: COBRA model
    nmr_profile: Series, index includes metabolite names used in exchange_map
    exchange_map: dict {metabolite_name: reaction_id}
    scale: converts NMR units to flux bounds (tunable)
    """
    m = base_model.copy()
    for met_name, rxn_id in exchange_map.items():
        if met_name not in nmr_profile.index:
            continue
        level = float(nmr_profile[met_name])
        rxn = m.reactions.get_by_id(rxn_id)
        # simple scheme: more concentration → more allowed uptake
        rxn.lower_bound = -scale * level
    return m

model_high_KER1 = apply_metabolite_constraints(model, nmr_high_mean, exchange_map)
model_low_KER1  = apply_metabolite_constraints(model, nmr_low_mean, exchange_map)


### 2.5: Build Model for KER 1 High and Low and Compare Fluxes

In [37]:

def run_fast_gimme(
    base_model,
    expr_series,
    nmr_profile,
    exchange_map,
    low_thresh=0.1,
    frac_of_opt=0.9,
    scale=0.01,
):
    """
    Fast GIMME:
      - Stage 1: maximise objective under NMR constraints
      - Stage 2: penalise only reactions active in Stage 1 AND low-expression
    """
    from optlang.symbolics import Zero

    # Apply NMR constraints
    m = apply_metabolite_constraints(base_model, nmr_profile, exchange_map, scale=scale)

    # Stage 1
    stage1_sol = m.optimize()
    if stage1_sol.status != "optimal":
        print("⚠ Stage 1 infeasible")
        return m, stage1_sol, None

    Zmax = stage1_sol.objective_value
    target = frac_of_opt * Zmax
    print(f"Stage 1 objective: {Zmax:.4f}, Stage 2 target: {target:.4f}")

    # Identify active reactions in Stage 1
    active_rxns = stage1_sol.fluxes[stage1_sol.fluxes.abs() > 1e-6].index

    # Build weights only for active + low-expression reactions
    weights = build_reaction_weights(m, expr_series, low_thresh=low_thresh)
    penalised_rxns = [rxn for rxn in active_rxns if weights.get(rxn, 0) > 0]

    # Add objective constraint
    orig_obj = m.objective.expression
    obj_constraint = m.problem.Constraint(orig_obj, lb=target, name="gimme_obj_constraint")
    m.add_cons_vars(obj_constraint)

    # Build new objective
    obj_expr = Zero
    for rxn_id in penalised_rxns:
        rxn = m.reactions.get_by_id(rxn_id)
        w = weights[rxn_id]
        obj_expr += w * rxn.forward_variable
        obj_expr += w * rxn.reverse_variable

    m.objective = m.problem.Objective(obj_expr, direction="min")

    # Stage 2
    gimme_sol = m.optimize()
    print("Stage 2 status:", gimme_sol.status)

    return m, stage1_sol, gimme_sol


# Run GIMME for KER1 High vs Low
# KER1 High
model_high_KER1_gimme, sol_high_stage1, sol_high_gimme = run_fast_gimme(
    model,
    expr_high_KER1,
    nmr_high_mean,
    exchange_map,
    low_thresh=0.1,
    frac_of_opt=0.9,
    scale=0.01,
)

# KER1 Low
model_low_KER1_gimme, sol_low_stage1, sol_low_gimme = run_fast_gimme(
    model,
    expr_low_KER1,
    nmr_low_mean,
    exchange_map,
    low_thresh=0.1,
    frac_of_opt=0.9,
    scale=0.01,
)

# Compare GIMME Fluxes for KER1 High vs Low
flux_high = sol_high_gimme.fluxes
flux_low  = sol_low_gimme.fluxes

flux_diff = flux_high - flux_low
flux_diff.name = "High_minus_Low"

flux_diff_sorted = flux_diff.reindex(flux_diff.abs().sort_values(ascending=False).index)

def show_reaction_changes(model, flux_diff, n=30, min_abs_change=0.0):
    sorted_ids = flux_diff.abs().sort_values(ascending=False).index
    count = 0
    for rxn_id in sorted_ids:
        change = flux_diff[rxn_id]
        if abs(change) < min_abs_change:
            continue
        rxn = model.reactions.get_by_id(rxn_id)
        print(f"{rxn_id:12s}  {change:8.3f}  {rxn.reaction}")
        count += 1
        if count >= n:
            break

show_reaction_changes(model, flux_diff, n=40, min_abs_change=0.01)



Stage 1 objective: 124.8681, Stage 2 target: 112.3813
Stage 2 status: optimal
Stage 1 objective: 124.8681, Stage 2 target: 112.3813
Stage 2 status: optimal
MAR05506      -1223.137  MAM02770c + MAM02896e <=> MAM02770e + MAM02896c
MAR09423      1003.532  MAM02996e <=> 
MAR11417      -1003.532  MAM02039e + MAM02996e <=> MAM02039c + MAM02996c
MAR04874      1000.991  MAM02578c <=> MAM02578m
MAR09068      -1000.002  MAM02770e <=> 
MAR03763      -1000.000  MAM01597m + MAM02039m + MAM02479m --> MAM00166m + MAM02040m
MAR09437      1000.000  MAM03118e <=> 
MAR04680      -1000.000  MAM00639c + MAM03118c <=> MAM01673c + MAM02751c
MAR04514      1000.000  MAM01668c + MAM02040c --> MAM01673c + MAM02578c
MAR06991      -1000.000  MAM02588c <=> MAM02588e
MAR04882      1000.000  MAM03118c <=> MAM03118e
MAR09092      1000.000  MAM01736e <=> 
MAR06001      1000.000  MAM01252c + MAM01253e <=> MAM01252e + MAM01253c
MAR13080      -1000.000  MAM02039i --> MAM02039m
MAR04979      -1000.000  MAM02997c --> MAM029